# LingGym Qwen2.5-7B Inference on Kaggle GPU

**Status**: Smoke test on Fwe language (147 questions)
**GPU**: T4 (16 GB VRAM)
**Model**: Qwen2.5-7B-Instruct with 4-bit quantization
**Expected Time**: 2-3 hours

## Cell 1: Install Dependencies

In [ ]:
# Install GPU-optimized packages
!pip install -q transformers torch accelerate bitsandbytes
!pip install -q huggingface-hub pandas tqdm

print("Dependencies installed successfully")

## Cell 2: Verify GPU and Setup

In [ ]:
import torch
import os
from pathlib import Path
import sys

# Verify GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Verify data directory
DATA_DIR = Path("/kaggle/input/linggym-benchmark/Benchmark_multiple_choice/Fwe")
print(f"\nData directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    files = list(DATA_DIR.glob("*.txt"))
    print(f"Question files found: {len(files)}")
    for f in sorted(files):
        print(f"  - {f.name}")

# Create output directory
OUTPUT_DIR = Path("/kaggle/output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"\nOutput directory ready: {OUTPUT_DIR}")

## Cell 3: Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

print("✓ Model loaded successfully")
print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB used")

## Cell 4: Load Questions

In [ ]:
import re
from pathlib import Path

DATA_DIR = Path("/kaggle/input/linggym-benchmark/Benchmark_multiple_choice/Fwe")

questions = []
for question_file in sorted(DATA_DIR.glob("*_questions.txt")):
    with open(question_file, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    
    q_count = content.count("Question ")
    questions.append({"file": question_file.name, "content": content})
    print(f"Loaded: {question_file.name} ({q_count} questions)")

total_q = sum(q["content"].count("Question ") for q in questions)
print(f"\nTotal questions: {total_q}")

## Cell 5: Run Inference

In [ ]:
from tqdm import tqdm
import gc

results = []
checkpoint_num = 0
CHECKPOINT_INTERVAL = 50

print("Starting inference...\n")

for file_data in questions:
    content = file_data["content"]
    file_name = file_data["file"]
    
    blocks = re.split(r"\n(?=Question \d+:)", content)
    print(f"Processing {file_name} ({len(blocks)} blocks)")
    
    for block in tqdm(blocks):
        if not block.strip().startswith("Question"):
            continue
        
        lines = block.split("\n")
        if len(lines) < 12:
            continue
        
        try:
            question_num = lines[0]
            prompt = "\n".join(lines[1:11])
            correct_line = lines[11] if len(lines) > 11 else ""
            correct_match = re.search(r"[A-D]", correct_line)
            correct_answer = correct_match.group(0) if correct_match else "?"
            
            # Generate prediction
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
            
            with torch.no_grad():
                generated_ids = model.generate(
                    **model_inputs,
                    do_sample=False,
                    max_new_tokens=32,
                )
            
            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(
                    model_inputs.input_ids, generated_ids
                )
            ]
            response = tokenizer.batch_decode(
                generated_ids, skip_special_tokens=True
            )[0]
            
            # Extract letter
            pred_match = re.search(r"[A-D]", response.upper())
            prediction = pred_match.group(0) if pred_match else "?"
            
            results.append({
                "file": file_name,
                "question_id": question_num,
                "prediction": prediction,
                "correct_answer": correct_answer,
                "match": prediction == correct_answer,
            })
            
            # Checkpoint
            if len(results) % CHECKPOINT_INTERVAL == 0:
                checkpoint_num += 1
                import pandas as pd
                pd.DataFrame(results).to_csv(
                    f"/kaggle/output/checkpoint_{checkpoint_num:03d}.csv"
                )
                gc.collect()
                torch.cuda.empty_cache()
        
        except Exception as e:
            results.append({
                "file": file_name,
                "question_id": question_num,
                "prediction": "ERROR",
                "correct_answer": "?",
                "match": False,
            })

print(f"\n✓ Inference complete. Processed {len(results)} predictions")

## Cell 6: Calculate Accuracy

In [ ]:
import pandas as pd
import json
from datetime import datetime

df = pd.DataFrame(results)

# Overall accuracy
total = len(df)
correct = df["match"].sum()
overall_accuracy = correct / total if total > 0 else 0

print(f"\nOVERALL RESULTS")
print(f"="*50)
print(f"Total Questions: {total}")
print(f"Correct Predictions: {correct}")
print(f"Accuracy: {overall_accuracy:.4f}")
print(f"\nPER-FILE RESULTS")
print(f"="*50)

summary = {
    "timestamp": datetime.now().isoformat(),
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "dataset": "LingGym Fwe",
    "total_questions": total,
    "correct_predictions": int(correct),
    "accuracy": float(overall_accuracy),
    "per_file": {}
}

for file_name in sorted(df["file"].unique()):
    file_df = df[df["file"] == file_name]
    file_correct = file_df["match"].sum()
    file_total = len(file_df)
    file_accuracy = file_correct / file_total if file_total > 0 else 0
    
    print(f"{file_name}: {file_accuracy:.4f} ({file_correct}/{file_total})")
    
    summary["per_file"][file_name] = {
        "total": int(file_total),
        "correct": int(file_correct),
        "accuracy": float(file_accuracy),
    }

## Cell 7: Save Results

In [ ]:
from pathlib import Path
import json

OUTPUT_DIR = Path("/kaggle/output")

# Save predictions CSV
csv_path = OUTPUT_DIR / "predictions.csv"
df.to_csv(csv_path, index=False)
print(f"✓ Predictions saved: {csv_path}")

# Save summary JSON
json_path = OUTPUT_DIR / "summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Summary saved: {json_path}")

# Save detailed report
report_path = OUTPUT_DIR / "detailed_report.txt"
with open(report_path, "w") as f:
    f.write("="*70 + "\n")
    f.write("LingGym Qwen2.5-7B Inference Results\n")
    f.write("="*70 + "\n\n")
    f.write(f"Timestamp: {summary['timestamp']}\n")
    f.write(f"Model: {summary['model']}\n")
    f.write(f"Dataset: {summary['dataset']}\n\n")
    f.write("OVERALL RESULTS\n")
    f.write("-"*70 + "\n")
    f.write(f"Total Questions: {summary['total_questions']}\n")
    f.write(f"Correct Predictions: {summary['correct_predictions']}\n")
    f.write(f"Accuracy: {summary['accuracy']:.4f}\n\n")
    f.write("PER-FILE RESULTS\n")
    f.write("-"*70 + "\n")
    for file_name, metrics in summary["per_file"].items():
        f.write(f"{file_name}\n")
        f.write(f"  Total: {metrics['total']}\n")
        f.write(f"  Correct: {metrics['correct']}\n")
        f.write(f"  Accuracy: {metrics['accuracy']:.4f}\n")

print(f"✓ Detailed report saved: {report_path}")
print("\n✓ Pipeline completed successfully")